# 06 — Clustering consolidado: zonas agroclimáticas y perfiles productivos

Este notebook reúne los **dos métodos de clustering que sí funcionaron** sobre
`dataset_integrado.csv` (2020–2025, 6 regiones): tipologías de **zona agroclimática**
(clima puro) y tipologías de **perfil productivo** (estacionalidad por cultivo).
Para cada uno se muestra: explicación breve → métricas (Silhouette, ARI) →
visualización geográfica de los clusters → gráficos complementarios.

Un tercer enfoque (mezclar clima + producción a nivel `(región, cultivo)`, más barridos
DBSCAN/PCA/NMF sobre filas mensuales) se probó y se descartó: el clima dominaba
artificialmente el agrupamiento y los barridos complementarios o tenían ruido inmanejable
(DBSCAN: 69.7%) o pseudo-replicaban datos mensuales. Ese notebook completo se conserva en
`BORRADORES/06_clustering_cultivos.ipynb`, con la razón del descarte documentada en su
primera celda — es evidencia de proceso, no un método vigente.

## Configuración común

In [1]:
# ====================================================================
# Importaciones y configuracion compartida por ambos metodos
# ====================================================================
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

import plotly.express as px
import plotly.figure_factory as ff

from scipy.cluster.hierarchy import linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
PALETTE = sns.color_palette("tab10")
print("Librerias cargadas")

Librerias cargadas


In [2]:
# ====================================================================
# Carga de dataset_integrado.csv y rutas de salida
# ====================================================================
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
elif ROOT.name == "Clustering":
    ROOT = ROOT.parent

RUTA_DATA = ROOT / "OUTPUTS" / "dataset_integrado.csv"
RUTA_OUT = ROOT / "OUTPUTS"
RUTA_FIG = RUTA_OUT / "figures"
RUTA_FIG.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RUTA_DATA)

CLIMA_VARS = [
    "temp_promedio", "temp_maxima", "temp_minima", "precipitacion",
    "humedad_relativa", "radiacion_solar", "velocidad_viento",
    "presion_atmosferica", "humedad_suelo", "temp_superficie",
    "punto_rocio", "humedad_especifica",
]
CLIMA_CORE = [
    "temp_promedio", "precipitacion", "humedad_relativa",
    "radiacion_solar", "humedad_suelo",
]

print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print("Regiones :", sorted(df["region"].unique()))
print("Distritos:", df["distrito"].nunique(), "| Pisos:", sorted(df["piso_ecologico"].unique()))
print("Cultivos :", df["cultivo"].nunique())

Dimensiones: 2,376 filas x 20 columnas
Regiones : ['Ica', 'Junin', 'La Libertad', 'Piura', 'Puno', 'San Martin']
Distritos: 28 | Pisos: ['altiplano_lacustre', 'bosque_seco', 'costa', 'puna_alta', 'selva_alta', 'selva_alto_mayo', 'selva_baja', 'selva_huallaga', 'sierra', 'valle_chira', 'valle_piura']
Cultivos : 19


In [3]:
# ====================================================================
# Funciones auxiliares de seleccion de K (reutilizables en ambos metodos)
# ====================================================================
def eval_kmeans_range(X, k_range, random_state=42):
    """Barrido de K: inercia, Silhouette, Calinski-Harabasz, Davies-Bouldin."""
    rows = []
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        lbl = km.fit_predict(X)
        rows.append({
            "k": k, "inertia": km.inertia_,
            "silhouette": silhouette_score(X, lbl),
            "calinski": calinski_harabasz_score(X, lbl),
            "davies_bouldin": davies_bouldin_score(X, lbl),
        })
    return pd.DataFrame(rows)


def elegir_k(df_km):
    """K optimo: coincidencia Silhouette/Davies-Bouldin, o ranking promedio."""
    best_sil = int(df_km.loc[df_km["silhouette"].idxmax(), "k"])
    best_db = int(df_km.loc[df_km["davies_bouldin"].idxmin(), "k"])
    if best_sil == best_db:
        return best_sil, "coincidencia Silhouette y Davies-Bouldin"
    rank_sil = df_km.set_index("k")["silhouette"].rank(ascending=False)
    rank_db = df_km.set_index("k")["davies_bouldin"].rank(ascending=True)
    k_opt = int((rank_sil + rank_db).idxmin())
    return k_opt, f"ranking promedio (Sil={best_sil}, DB={best_db})"


def titulo_centrado(texto, size=18):
    """Titulo de figura centrado y en negrita, mas prominente que los ejes."""
    return dict(text=f"<b>{texto}</b>", x=0.5, xanchor="center", font=dict(size=size))


NOMBRES_METRICA = {
    "inertia": "Inercia",
    "silhouette": "Silhouette",
    "davies_bouldin": "Davies-Bouldin",
}
COLOR_METRICA = {
    "Inercia": "#1f77b4",
    "Silhouette": "#2ca02c",
    "Davies-Bouldin": "#d62728",
}


def plot_k_selection(df_km, titulo):
    largo = df_km.melt(id_vars="k", value_vars=list(NOMBRES_METRICA),
                        var_name="metrica", value_name="valor")
    largo["metrica"] = largo["metrica"].map(NOMBRES_METRICA)
    fig = px.line(
        largo, x="k", y="valor", facet_col="metrica", color="metrica", markers=True,
        category_orders={"metrica": list(NOMBRES_METRICA.values())},
        color_discrete_map=COLOR_METRICA,
        template="plotly_white", facet_col_spacing=0.07,
    )
    fig.update_yaxes(matches=None, title_text="")
    fig.update_xaxes(title_text="Numero de clusters", dtick=1)
    # facet annotations vienen como "metrica=<texto>"; usar maxsplit=1 para no
    # cortar el texto si la etiqueta tiene su propio "=".
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=", 1)[-1], font=dict(size=12)))
    fig.update_layout(
        showlegend=False, width=1250, height=420,
        margin=dict(t=90, b=50, l=60, r=30),
        title=titulo_centrado(titulo, size=17),
    )
    fig.show()


def reporte_ari(labels_km, labels_hc, contexto):
    """Imprime el ARI entre KMeans y jerarquico como evidencia de robustez."""
    ari = adjusted_rand_score(labels_km, labels_hc)
    print(f"ARI KMeans vs Jerarquico ({contexto}): {ari:.3f} "
          f"({'alto acuerdo' if ari > 0.7 else 'acuerdo moderado/bajo'})")
    return ari

## A. Zonas agroclimáticas (clima puro)

**Objetivo:** agrupar las **zonas climáticas** (distrito proxy NASA) del territorio según
su clima 2020–2025. La producción **no** entra al clustering: es una variable
**descriptiva** que caracteriza cada zona *a posteriori*.

| Campo | Valor |
|-------|-------|
| Unidad de análisis | 1 fila por **distrito / zona climática** (28 zonas) |
| Input clustering | medias y desv. estándar climáticas (`CLIMA_CORE`) |
| Producción | descriptiva (boxplots por cluster), no input |

**Por qué esta versión:** en `dataset_integrado` el clima es constante por (región, piso),
de modo que las filas `(región, cultivo)` solo replican un número mucho menor de vectores
climáticos distintos. Aquí se clusteriza directamente la unidad real: **la zona
climática**. La interpretación es limpia: *tipologías agroclimáticas del territorio*.

In [4]:
# ====================================================================
# Tabla de zonas climaticas: 1 fila por distrito
# ====================================================================
agg_clima = {}
for v in CLIMA_CORE:
    agg_clima[f"{v}_mean"] = (v, "mean")
    agg_clima[f"{v}_std"] = (v, "std")

zonas = df.groupby(["distrito", "region", "piso_ecologico"], as_index=False).agg(**agg_clima)
zonas["etiqueta"] = zonas["region"] + " | " + zonas["distrito"]

FEATURES_ZONA = [c for c in zonas.columns if c.endswith("_mean") or c.endswith("_std")]
scaler = StandardScaler()
X_zona = scaler.fit_transform(zonas[FEATURES_ZONA])

print(f"Zonas climaticas: {len(zonas)} | features: {len(FEATURES_ZONA)}")

Zonas climaticas: 28 | features: 10


### A.1 Métricas: selección de K, Silhouette y robustez (ARI)

In [5]:
# ====================================================================
# KMeans sobre zonas climaticas + seleccion de K
# ====================================================================
km_df = eval_kmeans_range(X_zona, k_range=range(2, 7))
K_ZONA, motivo = elegir_k(km_df)
plot_k_selection(km_df, f"Seleccion de K - zonas (K={K_ZONA})")
print(f"K optimo: {K_ZONA} ({motivo})")

km_zona = KMeans(n_clusters=K_ZONA, random_state=42, n_init=10)
zonas["cluster"] = km_zona.fit_predict(X_zona)
sil_zona = silhouette_score(X_zona, zonas["cluster"])
db_zona = davies_bouldin_score(X_zona, zonas["cluster"])
print(f"KMeans K={K_ZONA} | Silhouette={sil_zona:.3f} | Davies-Bouldin={db_zona:.3f}")

K optimo: 6 (coincidencia Silhouette y Davies-Bouldin)
KMeans K=6 | Silhouette=0.498 | Davies-Bouldin=0.712


In [6]:
# ====================================================================
# Dendrograma Ward (validacion) + ARI KMeans vs Jerarquico
# ====================================================================
Z_zona = linkage(X_zona, method="ward")
thr_zona = (Z_zona[-K_ZONA, 2] + Z_zona[-(K_ZONA - 1), 2]) / 2

fig_dendro_zona = ff.create_dendrogram(
    X_zona, orientation="bottom", labels=zonas["etiqueta"].tolist(),
    linkagefun=lambda x: linkage(x, method="ward"), color_threshold=thr_zona,
)
fig_dendro_zona.add_hline(y=thr_zona, line_dash="dash", line_color="red",
                           annotation_text=f"Corte K={K_ZONA}", annotation_position="top right")
fig_dendro_zona.update_layout(
    title=titulo_centrado("Dendrograma Ward - zonas climaticas"),
    template="plotly_white", width=max(950, 42 * len(zonas)), height=600,
    margin=dict(t=80, b=180, l=70, r=30),
)
fig_dendro_zona.update_xaxes(tickangle=90, tickfont=dict(size=10),
                              title_text="Zona climatica (region | distrito)")
fig_dendro_zona.update_yaxes(title_text="Distancia (Ward)")
fig_dendro_zona.write_image(str(RUTA_FIG / "06a_dendrograma_zonas.png"), scale=2)
fig_dendro_zona.show()

hc_zona = AgglomerativeClustering(n_clusters=K_ZONA, linkage="ward")
zonas["cluster_jerarquico"] = hc_zona.fit_predict(X_zona)
ari_zona = reporte_ari(zonas["cluster"], zonas["cluster_jerarquico"], "zonas agroclimaticas")

ARI KMeans vs Jerarquico (zonas agroclimaticas): 0.829 (alto acuerdo)


### A.2 Visualización geográfica de los clusters

In [7]:
# ====================================================================
# Mapa interactivo de zonas-cluster (Folium)
# ====================================================================
try:
    import folium
    from branca.element import Element
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "folium==0.19.5"], check=True)
    import folium
    from branca.element import Element

COORDS_DISTRITO = pd.DataFrame([
    {"distrito": "Chincha Alta", "lat": -13.4099, "lon": -76.1324},
    {"distrito": "Viru", "lat": -8.4143, "lon": -78.7524},
    {"distrito": "Huamachuco", "lat": -7.8121, "lon": -78.0487},
    {"distrito": "Cascas", "lat": -7.4797, "lon": -78.8178},
    {"distrito": "Tambogrande", "lat": -4.9269, "lon": -80.3447},
    {"distrito": "Sullana", "lat": -4.9039, "lon": -80.6853},
    {"distrito": "Canchaque", "lat": -5.3760, "lon": -79.6098},
    {"distrito": "Moyobamba", "lat": -6.0344, "lon": -76.9742},
    {"distrito": "Tocache", "lat": -8.1877, "lon": -76.5205},
    {"distrito": "Perene", "lat": -10.9489, "lon": -75.2264},
    {"distrito": "Rio Tambo", "lat": -11.1656, "lon": -74.2353},
    {"distrito": "El Tambo", "lat": -12.0313, "lon": -75.2222},
    {"distrito": "Ilave", "lat": -16.0866, "lon": -69.6354},
    {"distrito": "Ayaviri", "lat": -14.8864, "lon": -70.5889},
])


def _hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255))


# Nombre "de cancha" + descripcion breve por cluster (ver tambien
# 07_analisis_clusters.ipynb, donde se valido cada nombre contra los cultivos
# y clima dominantes reales de cada cluster).
NOMBRES_ZONA = {
    0: "Oasis Piura",
    1: "Sierra papera",
    2: "Selva SM-Junin",
    3: "Puna Puno",
    4: "Costa canera",
    5: "Costa uvera",
}
NOMBRES_ZONA_DESC = {
    0: "Arroz, platano, mango y uva bajo riego en valles deserticos (Chira, Piura) "
       "y bosque seco. Riesgo: agua rio arriba y El Nino.",
    1: "Papa, alfalfa y avena forrajera para mercado interno y ganado lechero "
       "(Mantaro y sierra de La Libertad). Riesgo: heladas.",
    2: "Arroz, platano, cana, palma aceitera, pina y naranja en frontera amazonica "
       "(incluye Jilili-Piura, agrupado climaticamente con la selva). "
       "Riesgo: deforestacion y cambios de cultivo.",
    3: "Avena forrajera, alfalfa y papa que sostienen la ganaderia de alpaca/vacuno "
       ">3,800m. Riesgo: heladas que arruinan campanas enteras.",
    4: "Cana de azucar (outlier de Viru) y palta Hass con riego tecnificado. "
       "Riesgo: sequia extrema.",
    5: "Uva, arroz y esparrago que exigen humedad constante. "
       "Riesgo: falta de agua para riego continuo.",
}


zmap = zonas.merge(COORDS_DISTRITO, on="distrito", how="left")
colors_zona = {c: _hex(PALETTE[int(c) % len(PALETTE)]) for c in sorted(zonas["cluster"].unique())}

mapa_zonas = folium.Map(location=[-9.5, -75.5], zoom_start=5, tiles="CartoDB positron")

# Panel lateral fijo (en vez de popup-burbuja): se actualiza con JS al hacer
# clic en un punto. Empieza oculto hasta el primer clic.
mapa_zonas.get_root().html.add_child(Element("""
<div id="panel-zona" style="display:none;position:fixed;top:0;right:0;width:400px;height:100%;
     background:white;box-shadow:-6px 0 24px rgba(0,0,0,0.25);z-index:9999;
     font-family:sans-serif;overflow-y:auto;">
  <div id="panel-zona-header" style="padding:26px 24px;background:#4C78A8;color:white;position:relative;">
    <span onclick="document.getElementById('panel-zona').style.display='none'"
          style="position:absolute;top:18px;right:20px;cursor:pointer;font-size:26px;
                 line-height:1;font-weight:bold;">&times;</span>
    <div style="font-size:12px;opacity:0.85;letter-spacing:1.5px;text-transform:uppercase;">
      Cluster agroclimatico</div>
    <div id="panel-zona-titulo" style="font-size:23px;font-weight:bold;margin-top:10px;line-height:1.3;"></div>
  </div>
  <div style="padding:24px;">
    <div style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:6px;">
      Ubicacion</div>
    <div id="panel-zona-ubicacion" style="font-size:16px;margin-bottom:26px;color:#333;"></div>
    <div style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:6px;">
      Descripcion</div>
    <div id="panel-zona-desc" style="font-size:16px;line-height:1.7;color:#333;"></div>
  </div>
</div>
"""))

for _, row in zmap.iterrows():
    if pd.isna(row["lat"]):
        continue
    col = colors_zona[int(row["cluster"])]
    cl = int(row["cluster"])
    marker = folium.CircleMarker(
        location=[row["lat"], row["lon"]], radius=9, color=col, weight=2,
        fill=True, fill_color=col, fill_opacity=0.85,
        tooltip=f"C{cl} = {NOMBRES_ZONA[cl]} — {row['distrito']}",
    )
    marker.add_to(mapa_zonas)
    titulo = f"C{cl} = {NOMBRES_ZONA[cl]}".replace("'", chr(92) + "'")
    ubicacion = f"{row['region']} | {row['distrito']} ({row['piso_ecologico']})".replace("'", chr(92) + "'")
    desc = NOMBRES_ZONA_DESC[cl].replace("'", chr(92) + "'")
    # window 'load' difiere el binding hasta que el mapa y los marcadores ya
    # existan (Folium concatena todos los scripts en un solo bloque; sin el
    # 'load', esta linea se ejecutaba ANTES de que existiera el marcador y
    # rompia el resto del script, incluida la creacion del mapa).
    mapa_zonas.get_root().script.add_child(Element(f"""
        window.addEventListener('load', function() {{
            {marker.get_name()}.on('click', function(e) {{
                document.getElementById('panel-zona-header').style.background = '{col}';
                document.getElementById('panel-zona-titulo').innerHTML = '{titulo}';
                document.getElementById('panel-zona-ubicacion').innerHTML = '{ubicacion}';
                document.getElementById('panel-zona-desc').innerHTML = '{desc}';
                document.getElementById('panel-zona').style.display = 'block';
            }});
        }});
    """))

legend_zonas = "".join(
    f'<div style="margin-bottom:3px"><span style="background:{colors_zona[c]};width:14px;height:14px;'
    f'display:inline-block;margin-right:6px;border:1px solid #333;"></span>'
    f'C{c} = {NOMBRES_ZONA[c]}</div>'
    for c in sorted(zonas["cluster"].unique()))
mapa_zonas.get_root().html.add_child(Element(
    f'<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
    f'padding:10px;border:1px solid #ccc;border-radius:4px;font-size:12px;max-width:220px;">'
    f'<b>Zonas agroclimaticas</b><br>{legend_zonas}'
    f'<div style="margin-top:4px;color:#666;font-size:10px">Clic en un punto para ver ficha &rarr;</div>'
    f'</div>'))

ruta_mapa_zonas = RUTA_FIG / "06a_mapa_zonas.html"
mapa_zonas.save(str(ruta_mapa_zonas))
print(f"Mapa exportado: {ruta_mapa_zonas}")
mapa_zonas

Mapa exportado: C:\Users\Joyssie\Documents\dm\OUTPUTS\figures\06a_mapa_zonas.html


### A.3 Gráficos complementarios (clima y producción por cluster)

In [8]:
# ====================================================================
# Heatmap de centroides climaticos por cluster
# ====================================================================
cent_zona = zonas.groupby("cluster")[FEATURES_ZONA].mean()
cent_zona_norm = (cent_zona - cent_zona.min()) / (cent_zona.max() - cent_zona.min() + 1e-9)

fig_heat_zona = px.imshow(
    cent_zona_norm.T, text_auto=".2f", color_continuous_scale="YlOrRd", aspect="auto",
    labels=dict(x="Cluster", y="Variable", color="Normalizado"),
    x=[f"C{i}" for i in cent_zona.index], y=cent_zona_norm.columns,
)
fig_heat_zona.update_layout(title=titulo_centrado("Perfil climatico normalizado por cluster"),
                             template="plotly_white", margin=dict(t=80))
fig_heat_zona.write_image(str(RUTA_FIG / "06a_heatmap_zonas.png"), scale=2)
fig_heat_zona.show()

for c in sorted(zonas["cluster"].unique()):
    sub = zonas[zonas["cluster"] == c]
    print(f"\nCluster {c} ({len(sub)} zonas): {', '.join(sub['etiqueta'])}")
    print(f"  Pisos: {', '.join(sorted(sub['piso_ecologico'].unique()))}")
    print(f"  Temp media: {sub['temp_promedio_mean'].mean():.1f}C | "
          f"Precip: {sub['precipitacion_mean'].mean():.2f} | "
          f"Hum. suelo: {sub['humedad_suelo_mean'].mean():.2f}")


Cluster 0 (4 zonas): Piura | Castilla, Piura | Ignacio Escudero, Piura | Querecotillo, Piura | Tambogrande
  Pisos: bosque_seco, valle_chira, valle_piura
  Temp media: 25.6C | Precip: 0.49 | Hum. suelo: 0.41

Cluster 1 (5 zonas): La Libertad | Huamachuco, Junin | Huasahuasi, Junin | Matahuasi, Junin | San Agustin, Junin | San Jose de Quero
  Pisos: sierra
  Temp media: 8.9C | Precip: 0.97 | Hum. suelo: 0.47

Cluster 2 (9 zonas): San Martin | Bajo Biavo, Piura | Jilili, San Martin | Moyobamba, Junin | Pangoa, Junin | Perene, Junin | Pichanaqui, Junin | Rio Negro, Junin | Rio Tambo, San Martin | Tocache
  Pisos: selva_alta, selva_alto_mayo, selva_baja, selva_huallaga, sierra
  Temp media: 22.2C | Precip: 1.55 | Hum. suelo: 0.55

Cluster 3 (3 zonas): Puno | Ilave, Puno | Pucara, Puno | Taraco
  Pisos: altiplano_lacustre, puna_alta
  Temp media: 8.8C | Precip: 1.73 | Hum. suelo: 0.46

Cluster 4 (4 zonas): Ica | El Carmen, Ica | Independencia, Ica | Nazca, La Libertad | Viru
  Pisos: costa

In [9]:
# ====================================================================
# Produccion como variable descriptiva (a posteriori, no es input)
# ====================================================================
prod_zona = (df.groupby(["distrito", "cultivo"], as_index=False)["produccion_ton"].sum())
prod_zona = prod_zona.merge(zonas[["distrito", "cluster", "region"]], on="distrito", how="left")
prod_zona = prod_zona[prod_zona["produccion_ton"] > 0]

orden_clusters_zona = [f"C{c}" for c in sorted(zonas["cluster"].unique())]
fig_box_zona = px.box(
    prod_zona.assign(cluster_str="C" + prod_zona["cluster"].astype(str)),
    x="cluster_str", y="produccion_ton", log_y=True, points="outliers",
    category_orders={"cluster_str": orden_clusters_zona},
    labels={"cluster_str": "Cluster agroclimatico",
            "produccion_ton": "Produccion total 2020-2025 (t, log)"},
    template="plotly_white",
)
fig_box_zona.update_layout(title=titulo_centrado(
    "Produccion por cultivo dentro de cada zona-cluster (escala log)"), margin=dict(t=80))
# exponentformat="SI" evita mezclar numeros planos (2000000) con sufijos (10M) en el mismo eje
fig_box_zona.update_yaxes(exponentformat="SI", dtick=1)
fig_box_zona.write_image(str(RUTA_FIG / "06a_produccion_por_cluster.png"), scale=2)
fig_box_zona.show()

for c in sorted(zonas["cluster"].unique()):
    top = (prod_zona[prod_zona["cluster"] == c].groupby("cultivo")["produccion_ton"].sum()
           .sort_values(ascending=False).head(5))
    print(f"\nCluster {c} - cultivos dominantes:")
    for cult, val in top.items():
        print(f"  {cult:<22}: {val:,.0f} t")

zonas.to_csv(RUTA_OUT / "06a_zonas_clusters.csv", index=False, encoding="utf-8-sig")
prod_zona.to_csv(RUTA_OUT / "06a_produccion_por_zona.csv", index=False, encoding="utf-8-sig")
print("\nExportado: 06a_zonas_clusters.csv, 06a_produccion_por_zona.csv")


Cluster 0 - cultivos dominantes:
  arroz_cascara         : 2,860,924 t
  platano               : 2,091,358 t
  mango                 : 1,882,343 t
  uva                   : 1,465,184 t

Cluster 1 - cultivos dominantes:
  papa                  : 5,568,473 t
  alfalfa               : 829,310 t
  avena_forrajera       : 614,216 t
  maiz_choclo           : 540,869 t

Cluster 2 - cultivos dominantes:
  arroz_cascara         : 4,913,001 t
  platano               : 3,791,539 t
  cana_para_azucar      : 3,614,281 t
  palma_aceitera        : 3,021,208 t
  pina                  : 2,527,248 t

Cluster 3 - cultivos dominantes:
  avena_forrajera       : 13,294,902 t
  alfalfa               : 10,029,664 t
  papa                  : 4,821,686 t

Cluster 4 - cultivos dominantes:
  cana_para_azucar      : 28,614,989 t
  palta                 : 2,012,376 t
  mandarina             : 1,106,304 t
  maiz_amarillo_duro    : 924,061 t
  papa                  : 791,327 t

Cluster 5 - cultivos dominantes:
  uva

## B. Perfiles productivos por cultivo

**Objetivo:** agrupar las combinaciones `(región, cultivo)` por su **patrón de
producción** (forma estacional + escala + concentración), de modo que **el cultivo sí
determine el cluster** y no solo la geografía.

| Campo | Valor |
|-------|-------|
| Unidad de análisis | perfiles `(región, cultivo)` |
| Features dominantes | vector de **estacionalidad mensual** (12 fracciones) |
| Features de escala | `log1p(producción)`, n meses cosecha, concentración, CV |
| Clima | reducido a **2 ejes PCA** (peso menor, no domina) |

**Por qué esta versión:** el perfil estacional distingue cultivos *dentro* del mismo
distrito (cosecha concentrada vs. permanente), algo que el clima —constante por
distrito— no puede hacer. Así el clustering responde *"qué cultivos comparten patrón
productivo"*.

In [10]:
# ====================================================================
# Perfiles productivos: estacionalidad + escala + concentracion + clima (2 PCA)
# ====================================================================
MESES = list(range(1, 13))

piv = (df.groupby(["region", "cultivo", "numero_mes"])["produccion_ton"].sum()
       .unstack("numero_mes").reindex(columns=MESES).fillna(0.0))
totales = piv.sum(axis=1)
estacional = piv.div(totales.replace(0, np.nan), axis=0).fillna(0.0)
estacional.columns = [f"frac_m{m:02d}" for m in MESES]

perf = pd.DataFrame(index=piv.index)
perf["log_produccion"] = np.log1p(totales)
perf["frac_mes_pico"] = estacional.max(axis=1)
perf["n_meses_cosecha"] = (piv > 0).sum(axis=1)
mean_mes = piv.mean(axis=1)
perf["cv_mensual"] = piv.std(axis=1) / mean_mes.replace(0, np.nan)
perf["cv_mensual"] = perf["cv_mensual"].fillna(0.0)

clima_distrito = df.groupby(["region", "cultivo"])[CLIMA_CORE].mean()
clima_z = StandardScaler().fit_transform(clima_distrito)
clima_pca = PCA(n_components=2, random_state=42).fit_transform(clima_z)
perf["clima_pc1"] = clima_pca[:, 0]
perf["clima_pc2"] = clima_pca[:, 1]

df_perfil = estacional.join(perf).reset_index()
df_perfil["etiqueta"] = df_perfil["region"] + " | " + df_perfil["cultivo"]

FEATURES_PERFIL = list(estacional.columns) + [
    "log_produccion", "frac_mes_pico", "n_meses_cosecha", "cv_mensual",
    "clima_pc1", "clima_pc2",
]
X_perfil = StandardScaler().fit_transform(df_perfil[FEATURES_PERFIL])
print(f"Perfiles: {len(df_perfil)} | features: {len(FEATURES_PERFIL)} "
      f"({len(estacional.columns)} estacionales + {len(FEATURES_PERFIL)-len(estacional.columns)} escala/clima)")

Perfiles: 33 | features: 18 (12 estacionales + 6 escala/clima)


### B.1 Métricas: selección de K, Silhouette y robustez (ARI)

In [11]:
# ====================================================================
# KMeans sobre perfiles productivos + seleccion de K
# ====================================================================
km_df_perfil = eval_kmeans_range(X_perfil, k_range=range(2, 9))
K_PERFIL, motivo_perfil = elegir_k(km_df_perfil)
plot_k_selection(km_df_perfil, f"Seleccion de K - perfiles productivos (K={K_PERFIL})")
print(f"K optimo: {K_PERFIL} ({motivo_perfil})")

km_perfil = KMeans(n_clusters=K_PERFIL, random_state=42, n_init=10)
df_perfil["cluster"] = km_perfil.fit_predict(X_perfil)
sil_perfil = silhouette_score(X_perfil, df_perfil["cluster"])
db_perfil = davies_bouldin_score(X_perfil, df_perfil["cluster"])
print(f"KMeans K={K_PERFIL} | Silhouette={sil_perfil:.3f} | Davies-Bouldin={db_perfil:.3f}")

K optimo: 7 (ranking promedio (Sil=7, DB=8))
KMeans K=7 | Silhouette=0.308 | Davies-Bouldin=0.863


In [12]:
# ====================================================================
# Dendrograma Ward (validacion) + ARI KMeans vs Jerarquico
# ====================================================================
Z_perfil = linkage(X_perfil, method="ward")
thr_perfil = (Z_perfil[-K_PERFIL, 2] + Z_perfil[-(K_PERFIL - 1), 2]) / 2

fig_dendro_perfil = ff.create_dendrogram(
    X_perfil, orientation="bottom", labels=df_perfil["etiqueta"].tolist(),
    linkagefun=lambda x: linkage(x, method="ward"), color_threshold=thr_perfil,
)
fig_dendro_perfil.add_hline(y=thr_perfil, line_dash="dash", line_color="red",
                             annotation_text=f"Corte K={K_PERFIL}", annotation_position="top right")
fig_dendro_perfil.update_layout(
    title=titulo_centrado("Dendrograma Ward - perfiles productivos"),
    template="plotly_white", width=max(1050, 38 * len(df_perfil)), height=650,
    margin=dict(t=80, b=190, l=70, r=30),
)
fig_dendro_perfil.update_xaxes(tickangle=90, tickfont=dict(size=10),
                                title_text="Perfil (region | cultivo)")
fig_dendro_perfil.update_yaxes(title_text="Distancia (Ward)")
fig_dendro_perfil.write_image(str(RUTA_FIG / "06b_dendrograma_perfiles.png"), scale=2)
fig_dendro_perfil.show()

hc_perfil = AgglomerativeClustering(n_clusters=K_PERFIL, linkage="ward")
df_perfil["cluster_jerarquico"] = hc_perfil.fit_predict(X_perfil)
sil_perfil_hc = silhouette_score(X_perfil, df_perfil["cluster_jerarquico"])
print(f"Jerarquico K={K_PERFIL} | Silhouette={sil_perfil_hc:.4f}")
ari_perfil = reporte_ari(df_perfil["cluster"], df_perfil["cluster_jerarquico"], "perfiles productivos")

Jerarquico K=7 | Silhouette=0.3033
ARI KMeans vs Jerarquico (perfiles productivos): 0.754 (alto acuerdo)


### B.2 Visualización geográfica de los clusters

In [13]:
# ====================================================================
# Mapa de perfiles por region (Folium): cada (region, cultivo) coloreado por su cluster.
# La unidad no tiene coordenada propia (es region x cultivo, no distrito), asi que se
# distribuyen los perfiles de cada region alrededor de su capital en un pequeno circulo.
# ====================================================================
COORDS_REGION = {
    "Piura": (-5.1945, -80.6328),
    "La Libertad": (-8.1116, -79.0288),
    "Ica": (-14.0678, -75.7286),
    "San Martin": (-6.0344, -76.9742),
    "Junin": (-12.0651, -75.2049),
    "Puno": (-15.8402, -70.0219),
}

# Nombre "de cancha" + descripcion breve por cluster (ver tambien
# 07_analisis_clusters.ipynb, donde se valido cada nombre contra los cultivos
# y calendario de cosecha reales de cada cluster).
NOMBRES_PERFIL = {
    0: "Cierre de ano",
    1: "Invierno",
    2: "Todo el ano",
    3: "Ventana exportadora",
    4: "Todo o nada",
    5: "Papa atipica",
    6: "Otono andino",
}
NOMBRES_PERFIL_DESC = {
    0: "Tomate (Ica) y uva (Piura): pico nov-dic.",
    1: "Citricos, palta, cafe, papa, arroz, maiz amarillo y avena: pico may-jul, "
       "cierre de lluvias andinas e invierno costero.",
    2: "Yuca, platano, palma, cana, alfalfa, esparrago, naranja, pina, arroz y maiz "
       "(San Martin): perennes o bajo riego tecnificado, cosecha continua sin pico marcado.",
    3: "Uva (Ica) y mango (Piura): pico en enero para abastecer al Hemisferio Norte "
       "en su invierno.",
    4: "Arroz (La Libertad), avena y papa (Puno): un solo ciclo fuerte (abr-may); "
       "si falla, no hay plan B.",
    5: "Unico perfil de papa fuera de la sierra/altiplano (Ica), con pico en agosto "
       "— cluster de un solo elemento.",
    6: "Maiz choclo (Junin) y alfalfa (Puno): pico en abril, cierre de temporada de "
       "lluvias andina.",
}

df_perfil["orden_en_region"] = df_perfil.groupby("region").cumcount()
df_perfil["n_en_region"] = df_perfil.groupby("region")["region"].transform("size")

jitter_lat, jitter_lon = [], []
for _, row in df_perfil.iterrows():
    lat0, lon0 = COORDS_REGION[row["region"]]
    ang = 2 * np.pi * row["orden_en_region"] / max(row["n_en_region"], 1)
    radio = 0.35
    jitter_lat.append(lat0 + radio * np.sin(ang))
    jitter_lon.append(lon0 + radio * np.cos(ang))
df_perfil["lat"] = jitter_lat
df_perfil["lon"] = jitter_lon

colors_perfil = {c: _hex(PALETTE[int(c) % len(PALETTE)]) for c in sorted(df_perfil["cluster"].unique())}

mapa_perfiles = folium.Map(location=[-9.5, -75.5], zoom_start=5, tiles="CartoDB positron")

# Panel lateral fijo (en vez de popup-burbuja): se actualiza con JS al hacer
# clic en un punto. Empieza oculto hasta el primer clic.
mapa_perfiles.get_root().html.add_child(Element("""
<div id="panel-perfil" style="display:none;position:fixed;top:0;right:0;width:400px;height:100%;
     background:white;box-shadow:-6px 0 24px rgba(0,0,0,0.25);z-index:9999;
     font-family:sans-serif;overflow-y:auto;">
  <div id="panel-perfil-header" style="padding:26px 24px;background:#4C78A8;color:white;position:relative;">
    <span onclick="document.getElementById('panel-perfil').style.display='none'"
          style="position:absolute;top:18px;right:20px;cursor:pointer;font-size:26px;
                 line-height:1;font-weight:bold;">&times;</span>
    <div style="font-size:12px;opacity:0.85;letter-spacing:1.5px;text-transform:uppercase;">
      Cluster de perfil productivo</div>
    <div id="panel-perfil-titulo" style="font-size:23px;font-weight:bold;margin-top:10px;line-height:1.3;"></div>
  </div>
  <div style="padding:24px;">
    <div style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:6px;">
      Region y cultivo</div>
    <div id="panel-perfil-info" style="font-size:16px;margin-bottom:26px;color:#333;"></div>
    <div style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:0.5px;margin-bottom:6px;">
      Descripcion</div>
    <div id="panel-perfil-desc" style="font-size:16px;line-height:1.7;color:#333;"></div>
  </div>
</div>
"""))

for _, row in df_perfil.iterrows():
    col = colors_perfil[int(row["cluster"])]
    cl = int(row["cluster"])
    marker = folium.CircleMarker(
        location=[row["lat"], row["lon"]], radius=7, color=col, weight=2,
        fill=True, fill_color=col, fill_opacity=0.85,
        tooltip=f"C{cl} = {NOMBRES_PERFIL[cl]} — {row['cultivo']}",
    )
    marker.add_to(mapa_perfiles)
    titulo = f"C{cl} = {NOMBRES_PERFIL[cl]}".replace("'", chr(92) + "'")
    info = (f"{row['region']} | {row['cultivo']}<br>Mes pico: {row['frac_mes_pico']:.2f} | "
            f"Meses cosecha: {int(row['n_meses_cosecha'])}").replace("'", chr(92) + "'")
    desc = NOMBRES_PERFIL_DESC[cl].replace("'", chr(92) + "'")
    # window 'load' difiere el binding hasta que el mapa y los marcadores ya
    # existan (mismo motivo que en el mapa de zonas).
    mapa_perfiles.get_root().script.add_child(Element(f"""
        window.addEventListener('load', function() {{
            {marker.get_name()}.on('click', function(e) {{
                document.getElementById('panel-perfil-header').style.background = '{col}';
                document.getElementById('panel-perfil-titulo').innerHTML = '{titulo}';
                document.getElementById('panel-perfil-info').innerHTML = '{info}';
                document.getElementById('panel-perfil-desc').innerHTML = '{desc}';
                document.getElementById('panel-perfil').style.display = 'block';
            }});
        }});
    """))
for region, (lat0, lon0) in COORDS_REGION.items():
    folium.Marker(location=[lat0, lon0], icon=folium.Icon(color="gray", icon="info-sign"),
                  tooltip=region).add_to(mapa_perfiles)

legend_perfiles = "".join(
    f'<div style="margin-bottom:3px"><span style="background:{colors_perfil[c]};width:14px;height:14px;'
    f'display:inline-block;margin-right:6px;border:1px solid #333;"></span>'
    f'C{c} = {NOMBRES_PERFIL[c]}</div>'
    for c in sorted(df_perfil["cluster"].unique()))
mapa_perfiles.get_root().html.add_child(Element(
    f'<div style="position:fixed;bottom:24px;left:24px;z-index:9999;background:white;'
    f'padding:10px;border:1px solid #ccc;border-radius:4px;font-size:12px;max-width:220px;">'
    f'<b>Perfiles productivos por cluster</b><br>{legend_perfiles}'
    f'<div style="margin-top:4px;color:#666;font-size:10px">Clic en un punto para ver ficha &rarr;</div>'
    f'</div>'))

ruta_mapa_perfiles = RUTA_FIG / "06b_mapa_perfiles.html"
mapa_perfiles.save(str(ruta_mapa_perfiles))
print(f"Mapa exportado: {ruta_mapa_perfiles}")
mapa_perfiles

Mapa exportado: C:\Users\Joyssie\Documents\dm\OUTPUTS\figures\06b_mapa_perfiles.html


### B.3 Gráficos complementarios (estacionalidad y dispersión)

In [14]:
# ====================================================================
# Heatmap de estacionalidad media por cluster (12 meses)
# ====================================================================
nombres_mes = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
               "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
cent_est = df_perfil.groupby("cluster")[list(estacional.columns)].mean()

fig_heat_perfil = px.imshow(
    cent_est, text_auto=".2f", color_continuous_scale="Greens", aspect="auto",
    labels=dict(x="Mes", y="Cluster", color="Fraccion anual"),
    x=nombres_mes, y=[f"C{i}" for i in cent_est.index],
)
fig_heat_perfil.update_layout(
    title=titulo_centrado("Estacionalidad media por cluster (fraccion de produccion anual)"),
    template="plotly_white", margin=dict(t=80))
fig_heat_perfil.write_image(str(RUTA_FIG / "06b_heatmap_estacionalidad.png"), scale=2)
fig_heat_perfil.show()

for c in sorted(df_perfil["cluster"].unique()):
    sub = df_perfil[df_perfil["cluster"] == c]
    mes_pico = nombres_mes[int(cent_est.loc[c].values.argmax())]
    print(f"\nCluster {c} ({len(sub)} cultivos) | mes pico medio: {mes_pico}")
    print(f"  log_prod={sub['log_produccion'].mean():.1f} | "
          f"concentracion={sub['frac_mes_pico'].mean():.2f} | "
          f"meses_cosecha={sub['n_meses_cosecha'].mean():.1f}")
    print(f"  Cultivos: {', '.join(sub['region'] + '-' + sub['cultivo'])}")


Cluster 0 (2 cultivos) | mes pico medio: Dic
  log_prod=13.8 | concentracion=0.34 | meses_cosecha=12.0
  Cultivos: Ica-tomate, Piura-uva

Cluster 1 (9 cultivos) | mes pico medio: Jun
  log_prod=14.0 | concentracion=0.24 | meses_cosecha=11.1
  Cultivos: Ica-maiz_amarillo_duro, Ica-mandarina, Ica-palta, Junin-avena_forrajera, Junin-cafe_pergamino, Junin-papa, La Libertad-palta, La Libertad-papa, Piura-arroz_cascara

Cluster 2 (14 cultivos) | mes pico medio: Dic
  log_prod=14.5 | concentracion=0.13 | meses_cosecha=12.0
  Cultivos: Ica-alfalfa, Ica-esparrago, Junin-alfalfa, Junin-naranja, Junin-pina, Junin-platano, Junin-yuca, La Libertad-cana_para_azucar, Piura-cana_para_azucar, Piura-platano, San Martin-arroz_cascara, San Martin-maiz_amarillo_duro, San Martin-palma_aceitera, San Martin-platano

Cluster 3 (2 cultivos) | mes pico medio: Ene
  log_prod=14.6 | concentracion=0.39 | meses_cosecha=12.0
  Cultivos: Ica-uva, Piura-mango

Cluster 4 (3 cultivos) | mes pico medio: Abr
  log_prod=15

In [15]:
# ====================================================================
# PCA 2D de los perfiles, coloreado por cluster
# ====================================================================
pca_perfil = PCA(n_components=2, random_state=42)
xy_perfil = pca_perfil.fit_transform(X_perfil)
var_pc1, var_pc2 = pca_perfil.explained_variance_ratio_[:2] * 100

# Etiquetas interpretadas a partir de los loadings (ver celda de abajo): PC1 separa
# cosecha corta y concentrada en abr-may de cosecha extendida hacia oct-dic; PC2 separa
# pico de mitad de ano (jul-sep) de pico dic-ene muy concentrado en un solo mes.
ETIQUETA_PC1 = f"Componente 1 ({var_pc1:.0f}% var.): cosecha corta abr-may vs. extendida oct-dic"
ETIQUETA_PC2 = f"Componente 2 ({var_pc2:.0f}% var.): pico jul-sep disperso vs. pico dic-ene concentrado"

orden_clusters_perfil = [f"C{c}" for c in sorted(df_perfil["cluster"].unique())]
plot_pca = df_perfil.assign(
    pca1=xy_perfil[:, 0], pca2=xy_perfil[:, 1],
    cluster_str="C" + df_perfil["cluster"].astype(str),
)

fig_pca_perfil = px.scatter(
    plot_pca, x="pca1", y="pca2", color="cluster_str",
    hover_data=["region", "cultivo"],
    category_orders={"cluster_str": orden_clusters_perfil},
    labels={"pca1": ETIQUETA_PC1, "pca2": ETIQUETA_PC2, "cluster_str": "Cluster"},
    template="plotly_white",
    width=1000, height=750,
)
fig_pca_perfil.update_traces(marker=dict(size=11, line=dict(width=0.6, color="white")))
fig_pca_perfil.update_layout(
    title=titulo_centrado("PCA de perfiles productivos - KMeans"),
    legend=dict(title="Cluster"),
    margin=dict(t=90, b=60, l=70, r=30),
)
fig_pca_perfil.write_image(str(RUTA_FIG / "06b_pca_perfiles.png"), scale=2)
fig_pca_perfil.show()

# Loadings: que variables originales pesan mas en cada componente (sustento de las etiquetas)
loadings_perfil = pd.DataFrame(pca_perfil.components_.T, index=FEATURES_PERFIL, columns=["PC1", "PC2"])
print(f"Varianza explicada: PC1={var_pc1:.1f}% | PC2={var_pc2:.1f}%")
print("\nTop variables en PC1 (por |loading|):")
print(loadings_perfil["PC1"].reindex(loadings_perfil["PC1"].abs().sort_values(ascending=False).index).head(6))
print("\nTop variables en PC2 (por |loading|):")
print(loadings_perfil["PC2"].reindex(loadings_perfil["PC2"].abs().sort_values(ascending=False).index).head(6))

export_perfil = df_perfil[["region", "cultivo", "cluster", "cluster_jerarquico", "log_produccion",
                            "frac_mes_pico", "n_meses_cosecha", "cv_mensual"]].copy()
export_perfil.to_csv(RUTA_OUT / "06b_perfiles_productivos_clusters.csv",
                      index=False, encoding="utf-8-sig")
print("\nExportado: 06b_perfiles_productivos_clusters.csv")

Varianza explicada: PC1=32.6% | PC2=18.8%

Top variables en PC1 (por |loading|):
frac_m05           0.381453
frac_m04           0.365162
frac_m10          -0.330200
frac_m11          -0.301323
n_meses_cosecha   -0.281369
frac_m12          -0.280810
Name: PC1, dtype: float64

Top variables en PC2 (por |loading|):
frac_m07         0.395522
frac_m12        -0.342820
frac_m01        -0.316366
frac_m08         0.315168
frac_mes_pico   -0.301645
frac_m09         0.300544
Name: PC2, dtype: float64

Exportado: 06b_perfiles_productivos_clusters.csv


**Interpretación de los componentes:** PC1 (≈33% de la varianza) está dominado por las
fracciones de producción de abril-mayo (signo positivo) frente a octubre-noviembre-diciembre
y el número de meses de cosecha (signo negativo): separa cultivos con **pico fuerte en
abril-mayo y calendario corto** de cultivos con **cosecha extendida hacia fin de año**.
PC2 (≈19%) contrasta el pico en julio-agosto-septiembre (positivo) frente a diciembre-enero
y la concentración del pico mensual (`frac_mes_pico`, negativo): separa cultivos con
**pico a mitad de año** de los que cosechan a **fin/inicio de año o están muy concentrados
en un solo mes**. En conjunto, ambos ejes resumen sobre todo la **forma del calendario de
cosecha** (cuándo y qué tan concentrada es la producción) — coherente con que 12 de las 18
features de entrada son fracciones mensuales de estacionalidad, no escala ni clima.

## Comparación con el enfoque descartado (ARI vs. región)

Para auditar **qué** información captura cada partición se calcula el Índice de Rand
Ajustado (ARI) entre los clusters y la **región administrativa**: un ARI alto indica que
el modelo reproduce la geografía en vez de tipificar clima o patrón productivo de forma
genuina. Se compara contra el enfoque descartado (`BORRADORES/06_clustering_cultivos.ipynb`,
clima+producción mezclados a nivel `(región, cultivo)`), usando sus métricas ya exportadas
en `OUTPUTS/clustering_metricas.csv` y `OUTPUTS/clustering_perfiles.csv`.

In [16]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import adjusted_rand_score

metricas_descartado = pd.read_csv(RUTA_OUT / "clustering_metricas.csv")
sil_descartado = metricas_descartado.loc[
    metricas_descartado["configuracion"].str.startswith("Perfil KMeans"), "silhouette"].iloc[0]

perfiles_descartado = pd.read_csv(RUTA_OUT / "clustering_perfiles.csv")
ari_descartado_region = adjusted_rand_score(perfiles_descartado["cluster_kmeans"], perfiles_descartado["region"])
ari_zona_region = adjusted_rand_score(zonas["cluster"], zonas["region"])
ari_perfil_region = adjusted_rand_score(df_perfil["cluster"], df_perfil["region"])

comparativa = pd.DataFrame({
    "enfoque": ["06 Descartado<br>(clima+producción)", "A — Zonas<br>agroclimáticas", "B — Perfiles<br>productivos"],
    "silhouette": [sil_descartado, sil_zona, sil_perfil],
    "ari_region": [ari_descartado_region, ari_zona_region, ari_perfil_region],
})
print(comparativa.to_string(index=False))

colores_comp = ["#999999", "#1f77b4", "#2ca02c"]
fig_comparativa = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.15,
    subplot_titles=["Silhouette (mayor = mejor)", "ARI vs. región (menor = independiente de geografía)"],
)
fig_comparativa.update_annotations(font=dict(size=13))
fig_comparativa.add_trace(go.Bar(
    x=comparativa["enfoque"], y=comparativa["silhouette"], marker_color=colores_comp, showlegend=False,
    text=comparativa["silhouette"].round(3), textposition="outside"), row=1, col=1)
fig_comparativa.add_trace(go.Bar(
    x=comparativa["enfoque"], y=comparativa["ari_region"], marker_color=colores_comp, showlegend=False,
    text=comparativa["ari_region"].round(3), textposition="outside"), row=1, col=2)
fig_comparativa.update_yaxes(range=[0, max(comparativa["silhouette"].max(), 0.6) * 1.25], row=1, col=1)
fig_comparativa.update_yaxes(range=[0, max(comparativa["ari_region"].max(), 0.5) * 1.25], row=1, col=2)
fig_comparativa.update_layout(
    title=titulo_centrado("Comparación de los tres enfoques de clustering"),
    width=1100, height=520, margin=dict(t=140),
)
fig_comparativa.write_image(str(RUTA_FIG / "comparativa_3enfoques.png"), scale=2)
fig_comparativa.show()

                            enfoque  silhouette  ari_region
06 Descartado<br>(clima+producción)    0.427700    0.404141
        A — Zonas<br>agroclimáticas    0.498356    0.392463
        B — Perfiles<br>productivos    0.307668    0.040431


## Conclusión comparativa

| Criterio | **A — Zonas agroclimáticas** | **B — Perfiles productivos** |
|---|---|---|
| Unidad de análisis | 28 zonas (distrito) | perfiles `(región, cultivo)` |
| Qué entra al clustering | clima puro | estacionalidad + escala + clima (2 PCA) |
| ¿Qué separa los grupos? | Clima | Patrón de producción / calendario |
| Producción | descriptiva *a posteriori* | input dominante |
| Interpretación | limpia, sin perfiles duplicados | rica, pero exigente |

**Lectura conjunta:** ambos métodos son robustos en su propio espacio de features (ARI
KMeans-vs-Jerárquico alto en los dos casos, ver celdas A.1 y B.1), y responden preguntas
distintas y complementarias:

- **A** tipifica el **territorio**: qué climas hay y qué se cultiva en cada uno.
- **B** tipifica **patrones productivos de cultivos**: qué cultivos comparten calendario
  de cosecha, independientemente de su región.

El enfoque mixto (clima + producción a nivel `(región, cultivo)`, archivado en
`BORRADORES/06_clustering_cultivos.ipynb`) prometía clusterizar cultivos pero en la
práctica clusterizaba geografía, porque el clima —constante por distrito— dominaba el
espacio de features. Separar ambas preguntas (A y B) da resultados más honestos y
defendibles que mezclarlas.